In [ ]:
import json
import os
import warnings

import altair as alt
import matplotlib.pyplot as plt
import polars as pl
import tldextract

from src.utils import BASE_DIR, get_data_dir, setup_logging

warnings.filterwarnings("ignore", category=FutureWarning)

alt.data_transformers.enable("vegafusion")

setup_logging()

FORCE = False
DATE = "2025-10-11"
CLOUDFLARE_API_TOKEN = os.environ.get("CLOUDFLARE_API_TOKEN")

_, base_dir = get_data_dir(DATE)
splits_dir = base_dir / "splits"
files_dir = base_dir / "files"
analysis_dir = base_dir / "analysis"
plots_dir = BASE_DIR / "plots" / DATE

plots_dir.mkdir(parents=True, exist_ok=True)

## Calculate the overlap between different analyzed file sets

Before proceeding with the analysis, it is required to run the [following script](../scripts/find_files.sh) to generate the necessary analysis files.

In [ ]:
def load_domain_files_map() -> pl.DataFrame:
    """Load a mapping of domains to available policy files."""
    existing_map = analysis_dir / "domain_files_map.parquet"
    if existing_map.exists() and not (base_dir / "splits").exists():
        return pl.read_parquet(existing_map)

    split_files = list((base_dir / "splits").glob("split_*.txt"))
    if not split_files:
        return pl.read_parquet(existing_map)

    df_tmp = pl.concat(
        [
            pl.read_csv(f, has_header=False, new_columns=["domain"]).with_columns(pl.lit(f.stem).alias("split"))
            for f in split_files
        ]
    )

    files_to_join = [
        ("robots_urls.txt", "robots_txt"),
        ("ai_urls.txt", "ai_txt"),
        ("llms_urls.txt", "llms_txt"),
        ("llms-full_urls.txt", "llms_full_txt"),
    ]

    for filename, column_name in files_to_join:
        df_tmp = df_tmp.join(
            pl.read_csv(analysis_dir / "urls" / filename, has_header=False, new_columns=["domain"]).with_columns(
                pl.lit(True).alias(column_name)
            ),
            on="domain",
            how="left",
        )

    df_tmp = df_tmp.fill_null(False)

    return df_tmp


df_domain_files = load_domain_files_map()
df_domain_files.write_parquet(analysis_dir / "domain_files_map.parquet")

df_domain_files.head()

In [ ]:
df_domain_files.height

## Domain Categories

In [ ]:
raw_domain_details_path = analysis_dir / "cloudflare_domain_details.jsonl"
parquet_domain_details_path = analysis_dir / "cloudflare_domain_details.parquet"


crawl_domains = df_domain_files.select("domain").unique()

if not parquet_domain_details_path.exists() or FORCE:
    df_domain_details = (
        pl.read_ndjson(analysis_dir / "cloudflare_domain_details.jsonl").explode("result").unnest("result")
    )
    df_domain_details = df_domain_details.join(crawl_domains, on="domain", how="semi")

    df_domain_details.write_parquet(parquet_domain_details_path)
else:
    df_domain_details = pl.read_parquet(parquet_domain_details_path)
    df_domain_details = df_domain_details.join(crawl_domains, on="domain", how="semi")

df_domain_details.head()

In [ ]:
def build_category_mapping() -> pl.DataFrame:
    """Map Cloudflare category IDs to top-level category names."""
    categories = (
        df_domain_details.select("content_categories")
        .explode("content_categories")
        .unnest("content_categories")
        .drop_nulls("id")
        .select("id", "name", "super_category_id")
        .unique()
    )
    if categories.is_empty():
        return pl.DataFrame({"Category Name": ["Unknown"], "value": [-1]})

    parents = (
        categories.filter(pl.col("super_category_id").is_null())
        .select(pl.col("id").alias("parent_id"), pl.col("name").alias("parent_name"))
        .unique()
    )

    return pl.concat(
        [
            categories.with_columns(
                pl.coalesce([pl.col("super_category_id"), pl.col("id")]).alias("parent_id")
            )
            .join(parents, on="parent_id", how="left")
            .with_columns(pl.coalesce([pl.col("parent_name"), pl.col("name")]).alias("Category Name"))
            .select("Category Name", pl.col("id").alias("value"))
            .unique(),
            pl.DataFrame({"Category Name": ["Unknown"], "value": [-1]}),
        ]
    )


df_category_mapping = build_category_mapping()

df_category_mapping.head()

In [ ]:
def get_domain_categories() -> pl.DataFrame:
    """Map domains to their content categories."""
    df_tmp = (
        df_domain_details.explode("content_categories")
        .with_columns(pl.col("content_categories").struct.field("id"))
        .join(df_category_mapping, left_on="id", right_on="value")
        .group_by("domain")
        .agg(pl.col("Category Name").unique().alias("categories"))
        .explode("categories")
    )

    return pl.concat(
        [
            df_tmp,
            df_domain_details.filter(~pl.col("domain").is_in(df_tmp.get_column("domain").unique().to_list()))
            .with_columns(pl.lit("Unknown").alias("categories"))
            .select(pl.col("domain", "categories")),
        ]
    )


df_domain_categories = get_domain_categories()
df_domain_categories.head()

In [ ]:
def plot_domain_categories() -> alt.Chart:
    """Plot domain categories distribution.

    Results in a bar chart showing the number of domains per category, with 'Unknown' categories highlighted.
    """
    df_tmp = (
        df_domain_categories.group_by("categories")
        .len()
        .with_columns(pl.when(pl.col("categories") == "Unknown").then(1).otherwise(0).alias("type"))
    )

    chart = (
        alt.Chart(df_tmp)
        .mark_bar(size=15)
        .encode(
            x=alt.X("categories:N", sort="-y", title="Category", axis=alt.Axis(labelAngle=-45)),
            y=alt.Y("len:Q", title="Number of Domains"),
            color=alt.Color(
                "type:N",
                legend=None,
                scale=alt.Scale(domain=[0, 1], range=["steelblue", "gray"]),
            ),
        )
        .properties(
            width=600,
            height=400,
        )
    )

    chart.save(plots_dir / "domain_categories.pdf")
    chart.save(plots_dir / "domain_categories.png")
    return chart


plot_domain_categories()

In [ ]:
(
    df_domain_files.unpivot(
        on=["ai_txt", "llms_txt", "llms_full_txt", "robots_txt"],
        index="domain",
    )
    .join(
        df_domain_categories,
        on="domain",
    )
    .filter(pl.col("value"))
    .group_by("categories", "variable")
    .len()
    .join(
        df_domain_files.unpivot(
            on=["ai_txt", "llms_txt", "llms_full_txt", "robots_txt"],
            index="domain",
        )
        .filter(pl.col("value"))
        .group_by("variable")
        .len(),
        on="variable",
    )
    .with_columns((pl.col("len") / pl.col("len_right")).alias("percentage"))
)

In [ ]:
def plot_category_filetype_heatmap() -> alt.LayerChart:
    """Plot heatmap of file types per domain category.

    Results in a heatmap showing the percentage of domains in each category that have each file type.
    The percentage is calculated as the number of domains in a category with a file type divided by the total number of domains with that file type.
    """
    df_tmp = (
        df_domain_files.unpivot(
            on=["ai_txt", "llms_txt", "llms_full_txt", "robots_txt"],
            index="domain",
        )
        .join(
            pl.concat(
                [
                    df_domain_categories,
                    df_domain_details.filter(
                        ~pl.col("domain").is_in(df_domain_categories.get_column("domain").unique().to_list())
                    )
                    .with_columns(pl.lit("Unknown").alias("categories"))
                    .select(pl.col("domain", "categories")),
                ]
            ),
            on="domain",
        )
        .filter(pl.col("value"))
        .group_by("categories", "variable")
        .len()
        .join(
            df_domain_files.unpivot(
                on=["ai_txt", "llms_txt", "llms_full_txt", "robots_txt"],
                index="domain",
            )
            .filter(pl.col("value"))
            .group_by("variable")
            .len(),
            on="variable",
        )
        .with_columns((pl.col("len") / pl.col("len_right")).alias("percentage"))
    )

    base: alt.Chart = alt.Chart(
        df_tmp.join(
            # Calculate the order of categories based on overall frequency
            df_domain_categories.get_column("categories")
            .value_counts()
            .with_columns(
                pl.when(pl.col("categories") == "Unknown").then(pl.lit(-1)).otherwise(pl.col("count")).alias("count")
            )
            .sort("count", descending=True)
            .with_row_index(),
            left_on="categories",
            right_on="categories",
        )
    ).encode(
        x=alt.X("variable:N", title="File Type", axis=alt.Axis(labelAngle=-45)),
        y=alt.Y("categories:N", title="Category", sort=alt.EncodingSortField(field="index")),
    )

    heatmap = base.mark_rect().encode(
        color=alt.Color(
            "percentage:Q",
            title="Percentage",
            scale=alt.Scale(domainMin=0, domainMax=1, scheme="blues"),
        ),
    )

    # Reduce the font size for better readability
    text = base.mark_text(baseline="middle", fontSize=9).encode(
        text=alt.Text("percentage:Q", format=".2%"),
        color=alt.value("black"),
    )

    chart = alt.layer(heatmap, text).properties(
        width=200,
        height=450,
    )

    chart.save(plots_dir / "category_filetype_heatmap.pdf")
    chart.save(plots_dir / "category_filetype_heatmap.png")

    return chart


plot_category_filetype_heatmap()

### Subdomains

In [ ]:
def is_subdomain(domain: str) -> bool:
    """Check if a domain is a subdomain."""
    ext = tldextract.extract(domain)
    return ext.subdomain != ""


df_subdomain = df_domain_files.with_columns(pl.col("domain").map_elements(is_subdomain).alias("is_subdomain"))

df_subdomain = pl.concat(
    [
        df_subdomain.filter(pl.col("robots_txt") | pl.col("ai_txt") | pl.col("llms_txt") | pl.col("llms_full_txt"))
        .get_column("is_subdomain")
        .value_counts()
        .with_columns(pl.lit("domains_with_files").alias("group")),
        df_subdomain.get_column("is_subdomain").value_counts().with_columns(pl.lit("all_domains").alias("group")),
    ]
)


def plot_subdomain_proportion() -> alt.Chart:
    """Plot proportion of subdomains vs. root domains.

    Results in a normalized stacked bar chart showing the proportion of subdomains and root domains
    for all domains and for domains with files.
    """
    chart = (
        alt.Chart(df_subdomain)
        .mark_bar()
        .encode(
            x=alt.X("count", title="Count").stack("normalize"),
            y=alt.Y("group", title="Domains"),
            color=alt.Color("is_subdomain", title="Is Subdomain"),
        )
        .properties(width=400, height=300, title="Proportion of Subdomains vs. Root Domains")
        .resolve_scale(y="independent")
    )

    chart.save(plots_dir / "subdomain_proportion.pdf")
    chart.save(plots_dir / "subdomain_proportion.png")

    return chart


plot_subdomain_proportion()

In [ ]:
df_subdomain.with_columns(pl.col("count") / pl.col("count").sum().over("group").alias("total"))

### Top Level Domains

In [ ]:
def get_tld(domain: str) -> str:
    """Extract the top-level domain (TLD) from a URL."""
    ext = tldextract.extract(domain)
    return ext.suffix


file_name_mapping = {
    "robots_txt": "robots.txt",
    "ai_txt": "ai.txt",
    "llms_txt": "llms.txt",
    "llms_full_txt": "llms-full.txt",
}


df_tld = (
    df_domain_files.with_columns(pl.col("domain").map_elements(get_tld).alias("tld"))
    .unpivot(on=["robots_txt", "ai_txt", "llms_txt", "llms_full_txt"], index="tld")
    .filter(pl.col("value"))
)


def plot_top_tlds_per_file_type() -> alt.VConcatChart:
    """Plot top TLDs per file type.

    Results in a series of bar charts showing the top 10 TLDs for each file type,
    with an additional bar for 'Other' TLDs.
    """
    # Create individual charts for each variable
    charts = []
    for variable in ["robots_txt", "ai_txt", "llms_txt", "llms_full_txt"]:
        df_tmp = (
            df_tld.filter(pl.col("variable") == variable).group_by("tld").len().sort("len", descending=True).head(10)
        )

        # Calculate the count of the other TLDs
        other_count = df_tld.filter(pl.col("variable") == variable).height - df_tmp.select(pl.col("len")).sum()[0, 0]
        top_tld_counts = pl.concat(
            [
                df_tmp.with_columns(
                    [
                        pl.lit("Top 10").alias("type"),
                        pl.lit(0, dtype=pl.UInt32).alias("sort_order"),  # cast to UInt32
                    ]
                ),
                pl.DataFrame(
                    {"tld": ["other"], "len": [other_count], "type": ["Other"], "sort_order": [1]},
                    schema_overrides={"len": pl.UInt32, "sort_order": pl.UInt32},
                ),
            ]
        ).sort(["sort_order", "len"], descending=[False, True])

        chart = (
            alt.Chart(top_tld_counts)
            .mark_bar(size=15)
            .encode(
                x=alt.X(
                    "tld:N",
                    sort=None,  # Use dataframe order
                    axis=alt.Axis(labelAngle=-45),
                    title="TLDs",
                ),
                y=alt.Y(
                    "len:Q",
                    scale=alt.Scale(zero=True, domainMax=(top_tld_counts.select(pl.col("len")).max() * 1.1).item(0, 0)),
                    title="Count",
                ),
                color=alt.Color(
                    "type:N",
                    legend=None,
                    scale=alt.Scale(domain=["Top 10", "Other"], range=["steelblue", "gray"]),
                ),
            )
            .properties(width=200, height=150, title=file_name_mapping[variable])
        )

        charts.append(chart)

    # Combine charts with independent axes
    chart = alt.vconcat(alt.hconcat(charts[0], charts[1]), alt.hconcat(charts[2], charts[3])).resolve_scale(
        x="independent", y="independent"
    )

    chart.save(plots_dir / "top_tlds_per_file_type.pdf")
    chart.save(plots_dir / "top_tlds_per_file_type.png")

    return chart


plot_top_tlds_per_file_type()

In [ ]:
df_tld.group_by("tld").len().sort("len", descending=True).head(10)

### Most Common File Combinations

In [ ]:
from upsetplot import UpSet, from_indicators

upset_data = from_indicators(df_domain_files.select(["robots_txt", "ai_txt", "llms_txt", "llms_full_txt"]).to_dicts())

# Create UpSet plot with compact number formatting
fig = plt.figure()
upset = UpSet(upset_data, subset_size="count", show_counts="{:,d}", include_empty_subsets=True)
upset.plot(fig=fig)

fig.set_size_inches(12, 4, forward=True)

# Save the figure as PDF
fig.savefig(plots_dir / "upset_plot.pdf", bbox_inches="tight", dpi=300)
fig.savefig(plots_dir / "upset_plot.png", bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
# Calculate the percentage of domains with and without any file
df_domain_files.select(
    pl.any_horizontal(pl.col(["robots_txt", "ai_txt", "llms_txt", "llms_full_txt"])).alias("has_any_file")
).get_column("has_any_file").value_counts().with_columns((pl.col("count") / df_domain_files.height).alias("percentage"))

## Diff Permissions from Different Sources

Because the following scripts require `ripgrep` we can not execute them in the notebook directly. Instead, we run them as scripts and read the results here.

The following script generates a permissions map from the robots.txt and ai.txt files found in the dataset. Additionally, it diffs the permissions found in both files.

```bash
uv run src/compliance/map_permissions.py --date 2025-10-11 --data_dir ./data/ --force
```

The following script finds experimental directives related to AI training in the robots.txt and ai.txt files.

```bash
uv run src/compliance/find_experimental_directives.py --date 2025-10-11 --data_dir ./data/ --force
```

Run the relationship analysis after generating the permissions map.

```bash
uv run src/compliance/find_robots_ai_conficts.py --date 2025-10-11 --data_dir ./data/ --force
```


### Most Common Allowed and Disallowed Bots

In [ ]:
def parse_permission_map() -> list[dict[str, object]]:
    """Parse the permissions map from JSON file."""
    with (analysis_dir / "permissions_map.json").open("r") as f:
        permissions_map: dict[str, dict[str, dict[str, dict[str, list[str]]]]] = json.load(f)

    results: list[dict[str, object]] = []
    for url, value in permissions_map.items():
        for file_type in ["robots", "ai"]:
            if file_type not in value:
                continue
            for bot, permissions in value[file_type].items():
                for permission_type in ["allow", "disallow"]:
                    if permission_type in permissions and permissions[permission_type]:
                        results.append(
                            {
                                "url": url,
                                "bot": bot,
                                "file_type": file_type,
                                "permission_type": permission_type,
                            }
                        )

    return results


bot_mapping = pl.read_json(analysis_dir / "typo_mapping.json")
known_bots_path = analysis_dir / "cloudflare_known_bots.json"
if known_bots_path.exists():
    df_known_bots = pl.read_json(known_bots_path).explode("userAgentPatterns")
else:
    df_known_bots = pl.DataFrame(
        schema={
            "userAgentPatterns": pl.Utf8,
            "name": pl.Utf8,
            "operator": pl.Utf8,
            "category": pl.Utf8,
            "slug": pl.Utf8,
        }
    )

df_bot_details = (
    pl.DataFrame(parse_permission_map())
    .with_columns(
        pl.col("bot").replace(bot_mapping.get_column("original"), bot_mapping.get_column("corrected")).alias("bot")
    )
    .join(
        df_known_bots,
        left_on="bot",
        right_on="userAgentPatterns",
        how="left",
    )
)

df_bot_details.head()

In [ ]:
# Overall count of entries
df_bot_details.filter(pl.col("bot") != "*").height

In [ ]:
# Count of how many entries can not be mapped to a known bot name
df_bot_details.filter(pl.col("bot") != "*").get_column("name").null_count()

In [ ]:
# Count of unique bots without a known name (excluding wildcard entries)
df_bot_details.filter(pl.col("name").is_null() & (pl.col("bot") != "*")).get_column("bot").unique().len()

In [ ]:
def plot_top_bots_permissions() -> alt.VConcatChart:
    """Plot top bots by permissions.

    Results in a series of bar charts showing the top 10 bots for each file type and permission type,
    with an additional bar for 'Other' bots.
    """
    file_mapping = {
        "robots": "robots.txt",
        "ai": "ai.txt",
    }

    df_tmp = (
        df_bot_details.with_columns(
            pl.when(~pl.col("name").is_null()).then(pl.col("name")).otherwise(pl.col("bot")).alias("bot")
        )
        .group_by("bot", "file_type", "permission_type")
        .len()
        .sort("len", descending=True)
    )

    # Create individual charts for each variable
    charts = []
    for file_type in ["robots", "ai"]:
        for permission_type in ["allow", "disallow"]:
            df_filtered = df_tmp.filter(
                (pl.col("file_type") == file_type) & (pl.col("permission_type") == permission_type)
            ).head(10)

            # Calculate the count of the other bots
            total_count = df_tmp.filter(
                (pl.col("file_type") == file_type) & (pl.col("permission_type") == permission_type)
            ).height
            other_count = total_count - df_filtered.height if total_count > 10 else 0

            data = pl.concat(
                [
                    df_filtered.with_columns(
                        [
                            pl.lit("Top 10").alias("type"),
                            pl.lit(0, dtype=pl.UInt32).alias("sort_order"),
                        ]
                    ),
                    pl.DataFrame(
                        {
                            "bot": ["other"],
                            "file_type": [file_type],
                            "permission_type": [permission_type],
                            "len": [other_count],
                            "type": ["Other"],
                            "sort_order": [1],
                        },
                        schema_overrides={"len": pl.UInt32, "sort_order": pl.UInt32},
                    ),
                ]
            ).sort(["sort_order", "len"], descending=[False, True])

            chart = (
                alt.Chart(data)
                .mark_bar(size=15)
                .encode(
                    x=alt.X(
                        "bot:N",
                        title="Bots",
                        sort=None,
                        axis=alt.Axis(labelAngle=-45),
                    ),
                    y=alt.Y(
                        "len:Q",
                        title="Count",
                        scale=alt.Scale(zero=True, domainMax=(data.select(pl.col("len")).max() * 1.1).item(0, 0)),
                    ),
                    color=alt.Color(
                        "type:N",
                        legend=None,
                        scale=alt.Scale(domain=["Top 10", "Other"], range=["steelblue", "gray"]),
                    ),
                )
                .properties(width=225, height=150, title=f"{file_mapping[file_type]} - {permission_type}")
            )

            text = chart.mark_text(
                align="center",
                baseline="bottom",
                dy=-1,
            ).encode(text="len:Q")

            charts.append(chart + text)

    # Combine charts with independent axes
    chart = alt.vconcat(alt.hconcat(charts[0], charts[1]), alt.hconcat(charts[2], charts[3])).resolve_scale(
        x="independent", y="independent"
    )

    chart.save(plots_dir / "top_bots_permissions.pdf")
    chart.save(plots_dir / "top_bots_permissions.png")

    return chart


plot_top_bots_permissions()

In [ ]:
(
    df_bot_details.get_column("operator")
    .drop_nulls()
    .value_counts()
    .with_columns(
        pl.when(pl.col("count") < 75).then(pl.lit("Other")).otherwise(pl.col("operator")).alias("operator_grouped")
    )
    .group_by("operator_grouped")
    .agg(((pl.col("count").sum() / df_bot_details.drop_nulls().height) * 100).alias("percentage"))
    .sort("percentage", descending=True)
)

In [ ]:
def plot_bot_operator_distribution() -> alt.Chart:
    """Plot distribution of bot operators.

    Results in a normalized stacked bar chart showing the distribution of bot operators,
    grouping all operators with counts below a certain threshold into an 'Other' category.
    """
    df_tmp = (
        # Group all operators under the threshold into one category
        df_bot_details.get_column("operator")
        .drop_nulls()
        .value_counts()
        .with_columns(
            pl.when(pl.col("count") < 75).then(pl.lit("Other")).otherwise(pl.col("operator")).alias("operator_grouped")
        )
        .group_by("operator_grouped")
        .agg(pl.col("count").sum())
        .with_columns(pl.when(pl.col("operator_grouped") == "Other").then(pl.lit(0)).otherwise(pl.lit(1)).alias("type"))
        .sort(["type", "count"], descending=True)
        .with_row_index()
    )

    base = alt.Chart(df_tmp)

    chart = (
        base.mark_bar(size=30)
        .encode(
            x=alt.X("count:Q", title="Count").stack("normalize"),
            color=alt.Color(
                "operator_grouped:N", title="Bot Operator", sort=alt.EncodingSortField(field="index")
            ).scale(scheme="tableau20"),
            order=alt.Order("index:Q"),
        )
        .properties(width=400, height=40)
        .configure_legend(
            orient="bottom",
            columns=5,
        )
    )

    return chart


chart = plot_bot_operator_distribution()
chart.save(plots_dir / "bot_operator_distribution.pdf")
chart.save(plots_dir / "bot_operator_distribution.png")

chart

In [ ]:
(
    df_bot_details.get_column("category")
    .drop_nulls()
    .value_counts()
    .with_columns(
        pl.when(pl.col("count") < 75).then(pl.lit("Other")).otherwise(pl.col("category")).alias("category_grouped")
    )
    .group_by("category_grouped")
    .agg(((pl.col("count").sum() / df_bot_details.drop_nulls().height) * 100).alias("percentage"))
    .sort("percentage", descending=True)
)

In [ ]:
def plot_bot_category_distribution() -> alt.Chart:
    """Plot distribution of bot categories.

    Results in a normalized stacked bar chart showing the distribution of bot categories,
    grouping all categories with counts below a certain threshold into an 'Other' category.
    """
    base = alt.Chart(
        (
            # Group all categories under the threshold into one category
            df_bot_details.get_column("category")
            .drop_nulls()
            .value_counts()
            .with_columns(
                pl.when(pl.col("count") < 150)
                .then(pl.lit("Other"))
                .otherwise(pl.col("category"))
                .alias("category_grouped")
            )
            .group_by("category_grouped")
            .agg(pl.col("count").sum())
            .with_columns(
                pl.when(pl.col("category_grouped") == "Other").then(pl.lit(0)).otherwise(pl.lit(1)).alias("type")
            )
            .sort(["type", "count"], descending=True)
            .with_row_index()
        )
    )

    chart = (
        base.mark_bar(size=30)
        .encode(
            x=alt.X("count:Q", title="Proportion of Bots").stack("normalize"),
            color=alt.Color(
                "category_grouped:N",
                title="Bot Category",
                sort=alt.EncodingSortField(field="index"),
            ).scale(scheme="tableau20"),
            order=alt.Order("index:Q"),
        )
        .properties(width=400, height=40)
        .configure_legend(
            orient="bottom",
            columns=2,
            labelLimit=200,
        )
    )

    return chart


chart = plot_bot_category_distribution()
chart.save(plots_dir / "bot_category_distribution.pdf")
chart.save(plots_dir / "bot_category_distribution.png")

chart

In [ ]:
(
    pl.DataFrame(parse_permission_map())
    .group_by("bot", "file_type", "permission_type")
    .len()
    .sort("len", descending=True)
    .filter(pl.col("bot") != "*")
)

In [ ]:
# Optional: merge Cloudflare bot metadata with a local Vercel known-bots export if present.
import re

from bs4 import BeautifulSoup

vercel_known_bots_path = analysis_dir / "vercel_known_bots.html"
parsed_rows = []
if vercel_known_bots_path.exists():
    with vercel_known_bots_path.open("r") as f:
        html = f.read()

    soup = BeautifulSoup(html, "html.parser")
    table_body = soup.find("tbody")
    rows = table_body.find_all("tr") if table_body else []

    for row in rows:
        cells = row.find_all("td")
        bot_name = cells[0].get_text(strip=True)
        category = cells[1].get_text(strip=True)
        raw_desc = cells[2].get_text(" ", strip=True)
        description = re.sub(r"\s+", " ", raw_desc)
        documentation_link = cells[3].find("a")["href"] if cells[3].find("a") else None
        parsed_rows.append(
            {"bot_name": bot_name, "category": category, "description": description, "documentation": documentation_link}
        )

pl_vercel = pl.DataFrame(
    parsed_rows,
    schema={"bot_name": pl.Utf8, "category": pl.Utf8, "description": pl.Utf8, "documentation": pl.Utf8},
)

pl_vercel.head()

In [ ]:
df_known_bots.join(
    pl_vercel, left_on="slug", right_on="bot_name", how="full"
).head()

### Policy Directives

In [ ]:
df_directives = pl.read_parquet(analysis_dir / "exp_directives.parquet")

df_directives.head()

In [ ]:
df_directives.group_by("directive").len().sort("len", descending=True).head(20)

In [ ]:
df_directives.with_columns(
    pl.col("value").str.strip_chars().str.replace_all(r"(\s+)", "").str.split(",").list.sort()
).group_by("directive", "value").len().sort("len", descending=True).head(20)

In [ ]:
import matplotlib.patches as mpatches


def plot_directives_treemap() -> None:
    """Plot treemap of directives.

    Results in a treemap showing the distribution of Content-Signal and Content-Usage directives.
    The size of each rectangle is log-scaled to better visualize the distribution.
    """
    def split_treemap(items: list[dict], x: float, y: float, width: float, height: float) -> list[dict]:
        if len(items) == 1:
            return [items[0] | {"x": x, "y": y, "width": width, "height": height}]

        total = sum(item["size"] for item in items)
        midpoint = total / 2
        running = 0.0
        split_at = 1
        for index, item in enumerate(items):
            if index and running + item["size"] > midpoint:
                break
            running += item["size"]
            split_at = index + 1

        split_at = min(max(split_at, 1), len(items) - 1)
        left_items = items[:split_at]
        right_items = items[split_at:]
        left_total = sum(item["size"] for item in left_items)

        if width >= height:
            left_width = width * left_total / total
            return split_treemap(left_items, x, y, left_width, height) + split_treemap(
                right_items, x + left_width, y, width - left_width, height
            )

        left_height = height * left_total / total
        return split_treemap(left_items, x, y, width, left_height) + split_treemap(
            right_items, x, y + left_height, width, height - left_height
        )

    df_tmp = (
        df_directives.filter(pl.col("directive").is_in(["Content-Signal", "Content-signal", "Content-Usage"]))
        .with_columns(
            pl.col("directive").str.replace_all("Content-signal", "Content-Signal"),
            pl.col("value").str.strip_chars().str.split(","),
        )
        .explode("value")
        .with_columns(
            pl.col("value").str.strip_chars().str.split("=").list.get(0).alias("directive_name"),
            pl.col("value").str.strip_chars().str.split("=").list.get(1).alias("directive_value"),
        )
        .group_by("directive", "directive_name", "directive_value")
        .len()
        .with_columns((pl.col("len") + 1).log10().alias("size"))
        .sort("size", descending=True)
    )

    colors = {"Content-Signal": "#1f77b4", "Content-Usage": "#ff7f0e"}
    items = [
        {
            "label": f"{row['directive_name']}={row['directive_value']}",
            "directive": row["directive"],
            "count": row["len"],
            "size": row["size"],
            "color": colors.get(row["directive"], "#7f7f7f"),
        }
        for row in df_tmp.to_dicts()
    ]

    fig, ax = plt.subplots(figsize=(8, 6))
    for rect in split_treemap(items, 0, 0, 1, 1):
        x = rect["x"]
        y = rect["y"]
        width = rect["width"]
        height = rect["height"]
        ax.add_patch(
            mpatches.Rectangle((x, y), width, height, facecolor=rect["color"], edgecolor="white", linewidth=1.5)
        )

        if width * height > 0.015:
            ax.text(
                x + 0.01,
                y + height - 0.01,
                f"{rect['label']}\n{rect['count']:,}",
                ha="left",
                va="top",
                fontsize=9,
                color="white",
                clip_on=True,
            )

    legend_handles = [mpatches.Patch(color=color, label=directive) for directive, color in colors.items()]
    ax.legend(handles=legend_handles, loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.08), frameon=False)
    ax.set_axis_off()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    fig.tight_layout()

    fig.savefig(plots_dir / "content_signals_treemap.pdf", bbox_inches="tight")
    fig.savefig(plots_dir / "content_signals_treemap.png", bbox_inches="tight", dpi=300)
    plt.close(fig)


plot_directives_treemap()

In [ ]:
def plot_content_usage_directive() -> alt.FacetChart:
    """Plot Content-Usage directive details.

    Results in a faceted bar chart showing the counts of 'yes' and 'no' details for each purpose
    in the Content-Usage directive.
    """
    detail_mapping = {"y": "yes", "n": "no"}

    df_tmp = (
        df_directives.filter(pl.col("directive") == "Content-Usage")
        .with_columns(pl.col("value").str.split(", "))
        .explode("value")
        .with_columns(
            pl.col("value").str.split("=").list.get(0).alias("purpose"),
            pl.col("value").str.split("=").list.get(1).alias("detail"),
        )
        .group_by("purpose", "detail")
        .len()
        .filter(~(pl.col("detail") == "n;exceptions"))
        .with_columns(pl.col("detail").replace(detail_mapping))
    )

    bars = (
        alt.Chart(df_tmp)
        .mark_bar(size=15)
        .encode(
            x=alt.X("detail:N", title="Detail", sort=["yes", "no"], axis=alt.Axis(labelAngle=0)),
            y=alt.Y(
                "len:Q",
                title="Count",
                scale=alt.Scale(zero=True, domainMax=(df_tmp.select(pl.col("len")).max() * 1.1).item(0, 0)),
            ),
            color=alt.Color("detail:N", title="Detail", legend=None),
        )
    )

    text = bars.mark_text(
        align="center",
        baseline="bottom",
        dy=-1,
    ).encode(text="len:Q")

    chart = (
        alt.layer(bars, text)
        .properties(width=100, height=75)
        .facet(
            facet=alt.Facet("purpose:N", title="Purpose"),
            columns=3,
        )
    )

    chart.save(plots_dir / "content_usage_directive.pdf")
    chart.save(plots_dir / "content_usage_directive.png")

    return chart


plot_content_usage_directive()

In [ ]:
df_directives

In [ ]:
(
    df_directives.filter(pl.col("directive").str.contains("(?i)Content-Signal"))
    .with_columns(pl.col("value").str.split(","))
    .explode("value")
    .with_columns(
        pl.col("value").str.split("=").list.get(0).alias("purpose"),
        pl.col("value").str.split("=").list.get(1).alias("detail"),
    )
)

In [ ]:
def plot_content_signal_directive() -> alt.FacetChart:
    """Plot Content-Signal directive details.

    Results in a faceted bar chart showing the counts of 'yes' and 'no' details for each purpose
    in the Content-Usage directive.
    """
    df_tmp = (
        df_directives.filter(pl.col("directive").str.contains("(?i)Content-Signal"))
        .with_columns(pl.col("value").str.split(","))
        .explode("value")
        .with_columns(
            pl.col("value").str.split("=").list.get(0).str.strip_chars().alias("purpose"),
            pl.col("value").str.split("=").list.get(1).str.strip_chars().alias("detail"),
        )
        .group_by("purpose", "detail")
        .len()
        .filter(~(pl.col("detail") == "n;exceptions"))
    )

    bars = (
        alt.Chart(df_tmp)
        .mark_bar(size=15)
        .encode(
            x=alt.X("detail:N", title="Detail", sort=["yes", "no"], axis=alt.Axis(labelAngle=0)),
            y=alt.Y(
                "len:Q",
                title="Count",
                scale=alt.Scale(zero=True, domainMax=(df_tmp.select(pl.col("len")).max() * 1.1).item(0, 0)),
            ),
            color=alt.Color("detail:N", title="Detail", legend=None),
        )
    )

    text = bars.mark_text(
        align="center",
        baseline="bottom",
        dy=-1,
    ).encode(text="len:Q")

    chart = (
        alt.layer(bars, text)
        .properties(width=100, height=75)
        .facet(
            facet=alt.Facet("purpose:N", title="Purpose"),
            columns=3,
        )
    )

    chart.save(plots_dir / "content_signal_directive.pdf")
    chart.save(plots_dir / "content_signal_directive.png")

    return chart


plot_content_signal_directive()

### `robots.txt` and `ai.txt` Conflicts

In [ ]:
df_conflicts = pl.read_parquet(analysis_dir / "ai_conflicts.parquet")

df_conflicts.head()

In [ ]:
def plot_ai_robots_relationship() -> alt.LayerChart:
    """Plot relationship between ai.txt and robots.txt.

    Results in a bar chart showing the counts of different relationship types
    between ai.txt and robots.txt files.
    """
    df_tmp = df_conflicts.group_by("relationship").len()

    base = (
        alt.Chart(df_tmp)
        .mark_bar(size=15)
        .encode(
            x=alt.X("relationship:N", title="Relationship", axis=alt.Axis(labelAngle=-45)),
            y=alt.Y(
                "len:Q",
                title="Count",
                scale=alt.Scale(zero=True, domainMax=(df_tmp.select(pl.col("len")).max() * 1.1).item(0, 0)),
            ),
        )
    )

    text = base.mark_text(
        align="center",
        baseline="bottom",
        dy=-1,
    ).encode(text="len:Q")

    chart = alt.layer(base, text).properties(width=300, height=125)

    chart.save(plots_dir / "ai_robots_relationship.pdf")
    chart.save(plots_dir / "ai_robots_relationship.png")

    return chart


plot_ai_robots_relationship()

### `llms.txt` and `llms-full.txt` Conflicts

In [ ]:
df_blocked_paths = pl.read_parquet(analysis_dir / "llms_blocked_paths.parquet")

df_blocked_paths.head()

In [ ]:
# Calculate the number of llms.txt files that have a conflict in respect to the maximum number of llms.txt files
def calculate_blocked_path_percentage() -> dict[str, float]:
    """Calculate the percentage of llms.txt files with blocked paths."""
    restricted_llms_txt = df_blocked_paths.get_column("domain").unique().len()
    total = (
        pl.read_parquet(analysis_dir / "domain_files_map.parquet")
        .filter(pl.col("llms_txt") & (pl.col("robots_txt") | pl.col("ai_txt")))
        .height
    )

    return {
        "blocked_files": restricted_llms_txt,
        "all_files": total,
        "percentage": restricted_llms_txt / total,
    }


calculate_blocked_path_percentage()

In [ ]:
pl.read_parquet(analysis_dir / "domain_files_map.parquet").filter(
    pl.col("llms_txt") & (pl.col("robots_txt") | pl.col("ai_txt"))
).unpivot(
    on=["robots_txt", "ai_txt"],
    index="domain",
).filter(pl.col("value")).get_column("variable").value_counts().join(
    df_blocked_paths.group_by("domain", "blocking_file")
    .agg(pl.len().alias("num_blocked_paths"))
    .get_column("blocking_file")
    .value_counts(),
    left_on="variable",
    right_on="blocking_file",
).with_columns((pl.col("count_right") / pl.col("count")).alias("percentage"))

In [ ]:
pl.read_parquet(analysis_dir / "domain_files_map.parquet").filter(
    pl.col("llms_txt") & (pl.col("robots_txt") | pl.col("ai_txt"))
).unpivot(
    on=["robots_txt", "ai_txt", "llms_txt", "llms_full_txt"],
    index="domain",
)

### Duplicate Files

In [ ]:
import hashlib
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

from tqdm import tqdm


def file_hash(path: Path) -> tuple[str, str]:
    """Compute the SHA-256 hash of a file."""
    with path.open("rb") as f:
        return str(path), hashlib.sha256(f.read()).hexdigest()


hashes_file = analysis_dir / "file_hashes.parquet"

if not hashes_file.exists():
    files = [f for f in files_dir.rglob("*.txt") if "_invalid" not in f.parts]
    with ProcessPoolExecutor(max_workers=8) as executor:
        results = list(tqdm(executor.map(file_hash, files), total=len(files)))

    df_hashes = pl.DataFrame(results, schema=["path", "hash"], orient="row")
    df_hashes.write_parquet(hashes_file)
else:
    df_hashes = pl.read_parquet(hashes_file)

df_hashes.head()

In [ ]:
df_hashes.group_by("hash").agg(pl.len(), pl.col("path")).filter(pl.col("len") > 1).sort("len", descending=True).head(10)

In [ ]:
df_duplicates = (
    df_hashes.with_columns(
        pl.col("path").str.split("/").list.get(-2).alias("filename"),
        pl.col("path").str.split("/").list.get(-1).alias("filetype"),
    )
    .group_by("hash", "filetype")
    .agg(pl.col("path"), pl.len())
    .select(pl.col("hash", "filetype", "len"))
)

data = []

for file_type in df_duplicates.get_column("filetype").unique().to_list():
    df_type = df_duplicates.filter(pl.col("filetype") == file_type)
    total_files = df_type.select(pl.col("len")).sum()[0, 0]
    num_duplicates = df_type.filter(pl.col("len") > 1).height
    total_duplicate_files = df_type.filter(pl.col("len") > 1).select(pl.col("len")).sum()[0, 0]
    duplicate_percentage = total_duplicate_files / total_files if total_files > 0 else 0

    # Calculate files in each bucket
    files_1 = df_type.filter(pl.col("len") == 1).select(pl.col("len")).sum()[0, 0]
    files_2_10 = df_type.filter((pl.col("len") >= 2) & (pl.col("len") <= 10)).select(pl.col("len")).sum()[0, 0]
    files_11_100 = df_type.filter((pl.col("len") >= 11) & (pl.col("len") <= 100)).select(pl.col("len")).sum()[0, 0]
    files_over_100 = df_type.filter(pl.col("len") > 100).select(pl.col("len")).sum()[0, 0]

    data.append(
        {
            "file_type": file_type,
            "total_files": total_files,
            "num_duplicate_hashes": num_duplicates,
            "total_duplicate_files": total_duplicate_files,
            "duplicate_percentage": duplicate_percentage,
            "1": files_1 / total_files if total_files > 0 else 0,
            "2-10": files_2_10 / total_files if total_files > 0 else 0,
            "11-100": files_11_100 / total_files if total_files > 0 else 0,
            ">100": files_over_100 / total_files if total_files > 0 else 0,
        }
    )


df_duplicates_summary = pl.DataFrame(data)
df_duplicates_summary

In [ ]:
df_duplicates_summary.select(pl.col("file_type", "1", "2-10", "11-100", ">100")).with_columns(
    pl.col("1", "2-10", "11-100", ">100").round(4)
)